#### v1 jewelry_trends_scraper.py 22dec

#### v2 jewelry_trends_scraper_duckduckgo.py 22dec

 #### v3 agent.py  23dec

#### v4 agent_oxylabs.py 23dec

#### v5 agent_scrapegraph.py 23dec

#### v6 Duck Duck go, webcrawl without agno 24,25dec

In [ ]:
#!/usr/bin/env python3

from ddgs import DDGS
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import time

# -----------------------------
# CONFIG
# -----------------------------
USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
HEADERS = {"User-Agent": USER_AGENT}
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".webp")
MAX_SITES = 5
DELAY_BETWEEN_REQUESTS = 1.5  # seconds


# -----------------------------
# STEP 1: DuckDuckGo → WEBSITE URLs
# -----------------------------
def ddg_website_search(query: str, max_results: int = 5):
    urls = []
    with DDGS() as ddgs:
        results = ddgs.text(query, max_results=max_results)

        for r in results:
            if r.get("href"):
                urls.append(r["href"])

    return urls


# -----------------------------
# STEP 2: Visit website → extract image URLs
# -----------------------------
def extract_images_from_website(url: str):
    images = set()

    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()
    except Exception as e:
        print(f"[ERROR] Failed to fetch {url}: {e}")
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    # Find all <img> tags
    for img in soup.find_all("img"):
        src = img.get("src")
        if not src:
            continue

        full_url = urljoin(url, src)

        if full_url.lower().endswith(IMAGE_EXTENSIONS):
            images.add(full_url)

    return list(images)


# -----------------------------
# STEP 3: Main pipeline
# -----------------------------
def get_images_from_query(query: str):
    print(f"\n🔍 Searching websites for: {query}\n")

    websites = ddg_website_search(query, MAX_SITES)
    all_images = set()

    for site in websites:
        print(f"🌐 Crawling: {site}")
        images = extract_images_from_website(site)

        print(f"   ↳ Found {len(images)} images")
        for img in images:
            all_images.add(img)

        time.sleep(DELAY_BETWEEN_REQUESTS)

    return list(all_images)


# -----------------------------
# RUN
# -----------------------------
if __name__ == "__main__":
    query = "latest gold pendant jewelry designs"

    image_urls = get_images_from_query(query)

    print("\n✅ FINAL IMAGE URLs\n")
    for url in image_urls:
        print(url)

#### v7 agent agno_v6 26dec
Agno Agent (LLM)
→ decides refined search query + filters
→ calls deterministic tools
→ tools fetch websites
→ tools extract images
→ agent returns final image URLs

pip install agno duckduckgo-search requests beautifulsoup4


In [ ]:
from ddgs import DDGS
SOURCE= "giva.co"
def ddg_website_search(query: str, max_results: int = 5):
    """
    Search DuckDuckGo for website URLs.
    """
    urls = []
    with DDGS() as ddgs:
        results = ddgs.text(query, max_results=max_results)

        for r in results:
            if r.get("href"):
                urls.append(r["href"])

    return urls

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".webp")
HEADERS = {"User-Agent": "Mozilla/5.0"}


def extract_images_from_website(url: str):
    """
    Crawl a website and extract direct image URLs.
    """
    images = set()

    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()
    except Exception:
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    for img in soup.find_all("img"):
        src = img.get("src")
        if not src:
            continue

        full_url = urljoin(url, src)
        if full_url.lower().endswith(IMAGE_EXTENSIONS):
            images.add(full_url)

    return list(images)

from agno.agent import Agent
from agno.models.huggingface import HuggingFace
import os

HF_TOKEN = os.getenv("HF_TOKEN")

agent = Agent(
    name="Jewelry Image Intelligence Agent",
    model=HuggingFace(
        id="HuggingFaceTB/SmolLM3-3B",
        api_key=HF_TOKEN,
    ),
    tools=[
        ddg_website_search,
        extract_images_from_website
    ],
    instructions=[
        "Your job is to find jewelry pendant images.",
        "Step 1: Call ddg_website_search with a refined query.",
        "Step 2: For each returned website URL, call extract_images_from_website.",
        "Do not explain anything.",
        "Return only final image URLs.",
        "Do not hallucinate URLs."
    ],
    markdown=False
)

def agentic_image_pipeline(user_query: str):
    # Step 1: Agent decides search query
    refined_query_prompt = f"""
    Refine this query for discovering jewelry pendant websites:
    "{user_query}"
    Return only the refined query text.
    """

    refined_query = agent.run(refined_query_prompt).content
    if not refined_query:
        refined_query = user_query

    # Step 2: Get websites
    websites = ddg_website_search(refined_query, max_results=5)

    # Step 3: Crawl websites
    all_images = set()
    for site in websites:
        images = extract_images_from_website(site)
        for img in images:
            all_images.add(img)

    return list(all_images)


In [ ]:
if __name__ == "__main__":
    query = "latest gold butterfly pendant jewelry designs"

    image_urls = agentic_image_pipeline(query)

    print("\n✅ FINAL IMAGE URLs\n")
    for url in image_urls:
        print(url)



<span style="color: pink">The uncomfortable truth (said plainly)  
❌ Agentic ≠ Omniscient  
❌ An LLM cannot see images  
❌ It cannot judge relevance from a URL alone  
So when you crawl a website and extract <img> tags, you will always get:  
banners,logos,icons,tracking pixels,social buttons,unrelated images,.This is not an intelligence failure.It’s a missing perception layer.</span>

#### v8 Filtered img crawling on v7 27dec

Raw images
↓
Structural filters (fast, deterministic)
↓
Semantic filters (agentic reasoning)
↓
Visual filters (optional, best)

In [2]:
#!/usr/bin/env python3

import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from ddgs import DDGS

from agno.agent import Agent
from agno.models.huggingface import HuggingFace

# ================= CONFIG =================
HF_TOKEN = os.getenv("HF_TOKEN")

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120"
}

IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".webp")
BAD_KEYWORDS = [
    "logo", "icon", "sprite", "favicon", "avatar",
    "banner", "header", "footer", "ads"
]

# ================= AGENT =================
agent = Agent(
    name="Jewelry Pendant Image Agent",
    model=HuggingFace(
        id="HuggingFaceTB/SmolLM3-3B",
        api_key=HF_TOKEN
    ),
    instructions=[
        "You are a jewelry product expert.",
        "Select ONLY clear product or design images of jewelry pendants.",
        "Reject logos, UI elements, banners, people, lifestyle shots.",
        "Return ONLY valid image URLs.",
        "One URL per line. No explanation."
    ],
    markdown=False
)

# ================= HTTP SESSION =================
session = requests.Session()
session.headers.update(HEADERS)

# ================= SEARCH =================
def duckduckgo_website_search(query: str, max_results: int = 5):
    urls = []
    with DDGS() as ddgs:
        results = ddgs.text(query, max_results=max_results)
        for r in results:
            if r.get("href"):
                urls.append(r["href"])
    return urls

# ================= IMAGE EXTRACTION =================
def extract_images_with_context(url: str):
    images = []

    try:
        with session.get(url, timeout=10, stream=True) as response:
            response.raise_for_status()
            html = response.text
    except Exception:
        return images

    soup = BeautifulSoup(html, "html.parser")

    for img in soup.find_all("img"):
        src = img.get("src")
        if not src:
            continue

        full_url = urljoin(url, src)

        if not full_url.lower().endswith(IMAGE_EXTS):
            continue

        if any(bad in full_url.lower() for bad in BAD_KEYWORDS):
            continue

        alt = img.get("alt", "").strip()
        context = img.parent.get_text(" ", strip=True)[:300]

        images.append({
            "url": full_url,
            "alt": alt,
            "context": context
        })

    return images

# ================= AGENTIC FILTER =================
def agent_filter_images(images):
    if not images:
        return []

    prompt = f"""
From the following image data, select ONLY images that are jewelry pendants.

Rules:
- Must be a jewelry pendant product or design
- No logos, no UI, no banners, no people
- Ignore irrelevant objects

Return ONLY the image URLs.

Images:
{images}
"""

    response = agent.run(prompt)

    if response.content:
        return [
            line.strip()
            for line in response.content.splitlines()
            if line.strip().startswith("http")
        ]

    return []

# ================= PIPELINE =================
def get_relevant_pendant_images(query: str):
    websites = duckduckgo_website_search(query, max_results=5)

    collected_images = []
    for site in websites:
        collected_images.extend(extract_images_with_context(site))

    return agent_filter_images(collected_images)

# ================= RUN =================
if __name__ == "__main__":
    query = "latest gold butterfly pendant jewelry designs"

    results = get_relevant_pendant_images(query)

    print("\n✅ FINAL RELEVANT PENDANT IMAGE URLs\n")
    for url in results:
        print(url)


ERROR    HF_TOKEN not set. Please set the HF_TOKEN environment variable.

ERROR    Unexpected error invoking HuggingFace model: You must provide an api_key to work with auto API or log in  
         with `hf auth login`.

ERROR    Error in Agent run: You must provide an api_key to work with auto API or log in with `hf auth login`.


✅ FINAL RELEVANT PENDANT IMAGE URLs



#### v9 
27dec

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from agno.agent import Agent
from agno.models.ollama import Ollama
from ddgs import DDGS
from textwrap import dedent

# 1. Updated Scraper to handle multiple brands
def get_brand_jewelry_images(query: str, count: int =10) -> str:
    """
    Fetches jewelry image URLs from specific brands like Kalyan, Giva, and Palmonas.
    """
    results_list = []
    # List of targeted brands
    brands = ["Kalyan Jewelers", "GIVA Silver", "Palmonas"]

    try:
        with DDGS() as ddgs:
            # We construct a query that targets these specific sites
            # Example: "latest silver bangles (site:kalyanjewellers.net OR site:giva.co OR site:palmonas.com)"
            site_filter = "(site:kalyanjewellers.net OR site:giva.co OR site:palmonas.com OR site:pinterest.com)"
            # site_filter = "(site:giva.co OR site:palmonas.com OR site:pinterest.com)"

            combined_query = f"{query} {site_filter}"

            image_search = ddgs.images(
                query = combined_query,
                keywords=combined_query,
                region="wt-wt",
                safesearch="moderate",
                size="Medium",
                timelimit="d",
                type_image="photo",
                max_results=count
            )

            for r in image_search:
                # We return only the image link as per your request
                results_list.append(r['image'])

        return str(results_list) if results_list else "[]"
    except Exception as e:
        return f"Error: {str(e)}"

# 2. Setup the Agent
agent = Agent(
    model=Ollama(id="qwen2.5:7b"),
    tools=[get_brand_jewelry_images],
    instructions=dedent("""
       - You are a precise data assistant.
        - When the user asks for images, call 'get_jewelry_image_links'.
        - ONLY provide the raw URLs of the images.
        - Do not add descriptions or markdown formatting unless asked.
        - Please only output in json format.
    """),
    markdown=True
)

# 3. Execution
query = "gold ring"
agent.print_response(query)
res = agent.run(query).content
import json
res = json.loads(res)
print(res)

from IPython.display import Image, display

# Assuming 'res' is your list: ['url1', 'url2', ...]
print(f"Displaying {len(res)} images from the list:")

for link in res:
    try:
        # We use url=link to fetch the remote image
        # You can adjust width (e.g., 300) to make them smaller
        display(Image(url=link, width=400))
    except Exception as e:

        print(f"Could not load image from {link}: {e}")

/home/ec2-user/fusion_engine/.venv/lib64/python3.12/site-packages/rich/live.py:256: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

#### v10,selenium+Agno

In [6]:
#!/usr/bin/env python3

import os
import time
from dotenv import load_dotenv
load_dotenv()
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
from agno.agent import Agent
from agno.models.huggingface import HuggingFace

# ================= CONFIG =================
HF_TOKEN = os.getenv("HF_TOKEN")

IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".webp")
BAD_KEYWORDS = ["logo", "icon", "sprite", "favicon", "banner", "ads", "avatar"]

# ================= AGENT =================
agent = Agent(
    name="Jewelry Pendant Image Agent",
    model=HuggingFace(
        id="HuggingFaceTB/SmolLM3-3B",
        api_key=HF_TOKEN
    ),
    instructions=[
        "You are a jewelry product expert.",
        "Select ONLY clear product or design images of jewelry pendants.",
        "Reject logos, UI elements, banners, people, lifestyle shots.",
        "Return ONLY valid image URLs.",
        "One URL per line. No explanation."
    ],
    markdown=False
)

# ================= SELENIUM SETUP =================
def init_driver():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    driver = webdriver.Chrome(options=options)
    return driver

# ================= IMAGE EXTRACTION =================
def extract_images_from_page(driver, url, limit=20):
    driver.get(url)
    time.sleep(3)  # allow JS to load
    soup = BeautifulSoup(driver.page_source, "html.parser")

    images = []
    for img in soup.find_all("img"):
        src = img.get("src")
        if not src:
            continue
        if not src.lower().endswith(IMAGE_EXTS):
            continue
        if any(bad in src.lower() for bad in BAD_KEYWORDS):
            continue
        images.append(src)
        if len(images) >= limit:  # stop early
            break
    return images

# ================= AGENT FILTER =================
def agent_filter_images(images, max_results=5):
    if not images:
        return []
    prompt = f"""
From the following image data, select ONLY images that are jewelry pendants.

Rules:
- Must be a jewelry pendant product or design
- No logos, no UI, no banners, no people
- Ignore irrelevant objects

Return ONLY the image URLs.

Images:
{images}
"""
    response = agent.run(prompt)
    if response.content:
        urls = [
            line.strip()
            for line in response.content.splitlines()
            if line.strip().startswith("http")
        ]
        return urls[:max_results]  # enforce max 5
    return []

# ================= PIPELINE =================
def crawl_site(driver, site_name, collection_url):
    print(f"\n🔎 Crawling {site_name}...")
    raw_images = extract_images_from_page(driver, collection_url, limit=20)
    filtered = agent_filter_images(raw_images, max_results=5)
    return filtered

# ================= RUN =================
if __name__ == "__main__":
    driver = init_driver()

    sites = {
        "Giva": "https://www.giva.co/collections/pendants",
        "Kalyan": "https://www.kalyanjewellers.net/jewellery/pendants.php",
        "Malabar Golds": "https://www.malabargoldanddiamonds.com/jewellery/pendants.html"
    }

    all_results = {}
    for name, url in sites.items():
        all_results[name] = crawl_site(driver, name, url)

    driver.quit()

    print("\n✅ FINAL PENDANT IMAGE URLs (5 per site)\n")
    for site, urls in all_results.items():
        print(f"\n--- {site} ---")
        for u in urls:
            print(u)



🔎 Crawling Giva...

🔎 Crawling Kalyan...

🔎 Crawling Malabar Golds...

✅ FINAL PENDANT IMAGE URLs (5 per site)


--- Giva ---

--- Kalyan ---

--- Malabar Golds ---
https://static.malabargoldanddiamonds.com/media/catalog/category/hoops.jpg


#### v11 5 images+selenium+agno+no static image 29dec

In [8]:
#!/usr/bin/env python3

import os
from dotenv import load_dotenv
load_dotenv()
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
from agno.agent import Agent
from agno.models.huggingface import HuggingFace

# ================= CONFIG =================
HF_TOKEN = os.getenv("HF_TOKEN")

IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".webp")
BAD_KEYWORDS = ["logo", "icon", "sprite", "favicon", "banner", "ads", "avatar"]

# ================= AGENT =================
agent = Agent(
    name="Jewelry Pendant Image Agent",
    model=HuggingFace(
        id="HuggingFaceTB/SmolLM3-3B",
        api_key=HF_TOKEN
    ),
    instructions=[
        "You are a jewelry product expert.",
        "Select ONLY clear product or design images of jewelry pendants.",
        "Reject logos, UI elements, banners, people, lifestyle shots.",
        "Return ONLY valid image URLs.",
        "One URL per line. No explanation."
    ],
    markdown=False
)

# ================= SELENIUM SETUP =================
def init_driver():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    driver = webdriver.Chrome(options=options)
    return driver

# ================= IMAGE EXTRACTION =================
def extract_images(driver, url, selector, limit=20):
    driver.get(url)
    time.sleep(7)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(7)

    soup = BeautifulSoup(driver.page_source, "html.parser")
    images = []

    for img in soup.select(selector):
        src = img.get("src")
        if not src:
            continue
        if not src.lower().endswith(IMAGE_EXTS):
            continue
        if any(bad in src.lower() for bad in BAD_KEYWORDS):
            continue
        images.append(src)
        if len(images) >= limit:
            break

    return images

# ================= AGENT FILTER =================
def agent_filter_images(images, max_results=5):
    if not images:
        return []
    prompt = f"""
From the following image data, select ONLY images that are jewelry pendants.

Rules:
- Must be a jewelry pendant product or design
- No logos, no UI, no banners, no people
- Ignore irrelevant objects

Return ONLY the image URLs.

Images:
{images}
"""
    response = agent.run(prompt)
    if response.content:
        urls = [
            line.strip()
            for line in response.content.splitlines()
            if line.strip().startswith("http")
        ]
        return urls[:max_results]
    return []

# ================= PIPELINE =================
def crawl_site(driver, site_name, collection_url, selector):
    print(f"\n🔎 Crawling {site_name}...")
    raw_images = extract_images(driver, collection_url, selector, limit=20)
    filtered = agent_filter_images(raw_images, max_results=5)
    return filtered

# ================= RUN =================
if __name__ == "__main__":
    driver = init_driver()

    sites = {
        "Giva": {
            "url": "https://www.giva.co/collections/pendants",
            "selector": "img[src*='cdn/shop/files']"
        },
        "Kalyan": {
            "url": "https://www.kalyanjewellers.net/jewellery/pendants.php",
            "selector": "img.product-image"
        },
        "Malabar Golds": {
            "url": "https://www.malabargoldanddiamonds.com/jewellery/pendants.html",
            "selector": "img.product-image-photo"
        }
    }

    all_results = {}
    for name, profile in sites.items():
        all_results[name] = crawl_site(driver, name, profile["url"], profile["selector"])

    driver.quit()

    print("\n✅ FINAL PENDANT IMAGE URLs (5 per site)\n")
    for site, urls in all_results.items():
        print(f"\n--- {site} ---")
        for u in urls:
            print(u)



🔎 Crawling Giva...

🔎 Crawling Kalyan...

🔎 Crawling Malabar Golds...

✅ FINAL PENDANT IMAGE URLs (5 per site)


--- Giva ---

--- Kalyan ---

--- Malabar Golds ---
